# Week 4 Lecture: Data Wrangling in Python 🛠️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-04-lecture.ipynb)

Business questions almost never line up with how data is stored. Before you can say *which age group spends the most*, you have to clean up columns, collapse millions of rows into a few numbers, and pull information together from more than one table. That's the whole job this week: **manipulate, summarize, join**.

We'll warm up on the messy **Ames housing data**, then switch to **Complete Journey** — one year of real grocery purchases from 2,000+ households.

## How to use this notebook

This notebook accompanies the Week 4 Tuesday lecture. You can:

- **Follow along during class** — run each cell as we discuss it
- **Pause and experiment** — change the code and see what happens
- **Reference after class** — use it alongside the textbook chapters

## This Week's Topics

- **Chapter 10: Manipulating Data** — renaming, dropping, and creating columns, and handling missing values
- **Chapter 11: Aggregating Data** — summary statistics for a whole table and for each group
- **Chapter 12: Joining Data** — combining related tables with `.merge()`, and choosing a join type

## Setup

In [ ]:
# Complete Journey isn't preinstalled in Colab -- this line installs it (quick no-op if you already have it)
%pip install -q completejourney-py

In [ ]:
import pandas as pd
from completejourney_py import get_data

# Load every Complete Journey table into a dict of DataFrames
cj_data = get_data()
cj_data.keys()

## Meet the Complete Journey Data

**One year** of real grocery purchases from **2,000+ households**. Complete Journey isn't one big spreadsheet — it's **several related tables**. `transactions` records every item purchased, `products` describes each product, `demographics` describes the households that answered a survey, and so on. Shared columns like `product_id` and `household_id` connect them.

<img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/cj_data_relationships.png" width="500">

Docs: [bit.ly/completejourney_py](https://bit.ly/completejourney_py)

Run the cell below to see how big each table is.

In [ ]:
# How big is each table, and what columns does it have?
for name, df in cj_data.items():
    print(f"{name:22} {df.shape[0]:>12,} rows   {df.shape[1]} columns")

### Small Group Brainstorm

With your group:

1. Browse the **datasets** available in Complete Journey: [Complete Journey dataset guide](https://cunningjames.github.io/completejourney_py/user-guide/datasets/)
2. Write down **2–3 questions** a grocery retailer would want answered with this data.

For example:

- What income level is buying the most?
- Which department and product is the most commonly purchased?
- Which coupon was used the most?

**Your group's questions:**

1.
2.
3.

---

## Part 1: Manipulating Data 🔧

Real data rarely arrives ready for analysis. We'll use the raw Ames housing data to practice the four most common fixes: **cleaning column names**, **dropping columns**, **creating new columns**, and **handling missing values**.

### 🎯 What Would You Fix?

Import the raw Ames housing data:

In [ ]:
url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_raw.csv"
ames = pd.read_csv(url)

Spend a couple of minutes looking through `ames` with your Week 3 tools — `.head()`, `.columns`, `.info()`, `.describe()`.

**What would you want to change before you start an analysis?** Jot your ideas here:

-
-
-

In [ ]:
# Your code here

### Four Common Data Issues

> Datasets can be messy in **many** different ways. Today we'll illustrate **four common data issues** you'll often need to address.

1. **Inconsistent column names** — `MS SubClass`, `SalePrice`, `Year Remod/Add`
2. **Columns we don't need** — `Order`, `PID`
3. **Metrics that don't exist yet** — price per square foot?
4. **Missing values** — lots of `NaN`

In [ ]:
ames.head()

### Cleaning Up Column Names

To rename **a few** columns, give `rename()` a dict of `{old: new}` pairs:

In [ ]:
ames = ames.rename(columns={
    'MS SubClass': 'ms_subclass',
    'MS Zoning': 'ms_zoning'
})
ames.columns[:5]

With 82 columns, renaming one at a time isn't practical. `ames.columns` supports the same `.str` string methods as a text column, so you can clean **every** name in one chain:

In [ ]:
ames.columns = (
    ames.columns
    .str.lower()               # lowercase
    .str.replace(' ', '_')     # spaces  -> underscores
    .str.replace('/', '_')     # slashes -> underscores
)
ames.columns[:10]

> **⚠️ Assign it back — or nothing changes.** pandas methods **return a new result**; they don't change `ames` on their own. Run the cell below: it *shows* uppercase names, but `ames` itself is untouched.

In [ ]:
# This displays a changed copy... and then throws it away
ames.columns.str.upper()[:3]

In [ ]:
# ...so ames still has the lowercase names from before
ames.columns[:3]

### Selecting vs. Dropping Columns

Last week you **selected** the columns you wanted. Sometimes it's easier to **drop** the ones you don't. Rule of thumb: keeping a **few** columns → select. Removing a **few** columns → drop.

In [ ]:
# Keep columns of interest (just a view -- ames is unchanged)
cols = ['neighborhood', 'gr_liv_area', 'saleprice']
ames[cols].head(3)

In [ ]:
# Drop columns of disinterest -- and assign it back
print(f"Before: {ames.shape}")
ames = ames.drop(columns=['order', 'pid'])
print(f"After:  {ames.shape}")

### Creating New Columns

The pattern is always the same: **`df['new_col'] = <something computed from other columns>`**. You can build a new metric with arithmetic...

In [ ]:
# Price per square foot of living area
ames['price_per_sqft'] = ames['saleprice'] / ames['gr_liv_area']

ames[['saleprice', 'gr_liv_area', 'price_per_sqft']].head(3)

...or translate values with `.map()` and a dict of `{old value: new value}`:

In [ ]:
months = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
          5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
          9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}

ames['month_sold'] = ames['mo_sold'].map(months)
ames[['mo_sold', 'month_sold']].head(3)

### Handling Missing Values

**Step 1 — Find it.** `.isnull()` marks every missing cell as `True`, and `.sum()` counts them per column:

In [ ]:
ames.isnull().sum().sort_values(ascending=False).head()

**Step 2 — Ask *why*.** Almost every home is missing `pool_qc` (pool quality). Is that a data-entry problem, or does it mean something?

In [ ]:
missing_qc = ames['pool_qc'].isnull().sum()
no_pool = (ames['pool_area'] == 0).sum()

print(f"Homes missing pool_qc:    {missing_qc:,}")
print(f"Homes with pool_area = 0: {no_pool:,}")

**Step 3 — Fill it with what it actually means.** Those numbers match: `NaN` here means *"there's no pool."* So the right fix is a label — not deleting rows or inventing an average.

In [ ]:
ames['pool_qc'] = ames['pool_qc'].fillna('no pool')
ames['pool_qc'].value_counts()

> **Missing isn't always an error.** Understand *why* data is missing before you decide how to handle it.

### 🎯 Your Turn: Clean It Up

Back to Complete Journey. Using the `products` and `transactions` tables loaded below:

1. In `products`, create a `clean_category` column: `product_category` in **lowercase** with spaces replaced by **underscores**.
    - *Optional:* some products are missing a category. See if you can fill these in with `"unknown"`.
2. In `transactions`, create a `unit_price` column: `sales_value` divided by `quantity`.
3. Sort `transactions` by `unit_price`, highest first. **What looks strange?**

In [ ]:
products = cj_data['products']
transactions = cj_data['transactions']

In [ ]:
# Your code here

---

## Part 2: Summarizing Data 📊

`transactions` has **1.4 million rows**. Nobody wants to read them — they want answers: *What are total sales? Which households spend the most? Which stores bring in the most revenue?* Most business questions are **aggregate** questions — they collapse many rows into a few numbers, often **one number per group**.

<img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/summarizing-by-groups.png" width="600">

### Summarizing the Whole Table

One statistic for one column:

In [ ]:
total_sales = transactions['sales_value'].sum()
avg_sales = transactions['sales_value'].mean()

print(f"Total sales:              ${total_sales:,.2f}")
print(f"Average item sales value: ${avg_sales:,.2f}")

Several statistics for several columns → `.agg()` with a dict of **`{column: statistic(s)}`**. Common statistics: `'sum'`, `'mean'`, `'median'`, `'min'`, `'max'`, `'count'`, `'nunique'`.

In [ ]:
transactions.agg({
    'sales_value': ['sum', 'mean'],
    'quantity': ['sum', 'max']
})

### The Groupby Model

Grouped aggregation always follows the same three steps:

1. **Split** the rows into groups — `.groupby('store_id')`
2. **Apply** a summary to each group — `.agg({'sales_value': 'sum'})`
3. **Combine** the results into a new DataFrame — one row per group

<img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/model-for-grouped-aggs.png" width="600">

`as_index=False` keeps the group labels as a normal column instead of the index.

In [ ]:
# Total sales for each store
transactions.groupby('store_id', as_index=False).agg({'sales_value': 'sum'}).head()

### Group → Aggregate → Sort

**Business Question:** Who are our top 5 households by total spend — and how many shopping trips did it take them?

`'nunique'` counts **distinct** values. A basket with 20 items is still **one** trip, so counting unique `basket_id`s gives the number of trips.

The result is a *count* of baskets, not a basket ID — so `.rename()` it to say what it holds.

In [ ]:
(
    transactions
    .groupby('household_id', as_index=False)
    .agg({'sales_value': 'sum', 'basket_id': 'nunique'})
    .rename(columns={'basket_id': 'n_baskets'})   # it's a count now, not an ID
    .nlargest(5, 'sales_value')
)

Want one group for every **combination** of two columns? Pass a list to `.groupby()`:

In [ ]:
# Spend for each household at each store they shop at
(
    transactions
    .groupby(['household_id', 'store_id'], as_index=False)
    .agg({'sales_value': 'sum'})
    .head()
)

### 🎯 Your Turn: Summarize It

Using `transactions`:

1. Find the **top 10 products** (`product_id`) by **total** `sales_value`.
2. For each **store**, compute **total sales** and the **number of unique baskets**. Which store brings in the most revenue?
3. Look back at your answer to #1. **Anything surprising about the #1 product?**

In [ ]:
# Your code here

---

## Part 3: Joining Data 🔗

### What Is Product 6534178?

If you finished the last challenge, one product brings in about **10× more** than any other. But `transactions` only stores the ID — the product details live in a **different table**.

Organizations store data in separate tables because it's more efficient, and because different teams collect different pieces. To answer a question that spans tables, we **join** them on a **key**: a column that appears in both.

| Key | Connects |
|-----|----------|
| `product_id` | `transactions` ↔ `products` |
| `household_id` | `transactions` ↔ `demographics` |
| `coupon_upc` | `coupons` ↔ `coupon_redemptions` |

A key needs the **same meaning and the same data type** in both tables — otherwise rows fail to match, or match wrong.

<img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/cj_data_relationships.png" width="500">

In [ ]:
# The same key column, in two different tables
print(transactions[['product_id', 'sales_value']].head(3), end="\n\n")
print(products[['product_id', 'department', 'product_type']].head(3))

### Four Types of Joins

The join type decides **what happens to rows that don't find a match**:

| `how=` | Keeps |
|--------|-------|
| `'inner'` | Only rows with a match in **both** tables |
| `'left'` | **All** rows from the left table, plus matches from the right |
| `'right'` | **All** rows from the right table, plus matches from the left |
| `'outer'` | **All** rows from **both** tables |

| Inner | Left | Right | Outer |
|:-----:|:----:|:-----:|:-----:|
| <img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/join-inner.png" width="200"> | <img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/join-left.png" width="200"> | <img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/join-right.png" width="200"> | <img src="https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/slides/images/join-outer-full.png" width="200"> |

### Joining with `.merge()`

```python
# how = 'inner', 'left', 'right', or 'outer'
left_df.merge(right_df, on='key_column', how='inner')
```

> **Tip:** `pd.merge(left_df, right_df, on='key_column', how='inner')` does the same thing — you'll see both.

**Mystery solved:** summarize first, then join on the product details.

In [ ]:
# Step 1 -- the top 5 products by total sales (still just IDs)
top_products = (
    transactions
    .groupby('product_id', as_index=False)
    .agg({'sales_value': 'sum'})
    .nlargest(5, 'sales_value')
)

# Step 2 -- keep only the product columns we need, then join them on
product_info = products[['product_id', 'department', 'product_type']]
top_products.merge(product_info, on='product_id', how='left')

Four of our top five "products" are **gasoline** — the kiosk at the store. Our real grocery best seller is a gallon of milk. **A join turned an ID into an insight.**

> **Tip:** joining the small 5-row summary is much faster than joining all 1.4 million transactions first. Summarize first when you can.

### 🎯 Predict It

- `transactions` has **1,469,307** rows from **2,469** households.
- `demographics` has one row for each of **801** households.

If we join them on `household_id`, **how many rows come back?**

1. With an **inner** join?
2. With a **left** join (`transactions` on the left)?

**Write down your guesses first** — then run both joins below and check the row counts with `len()`.

In [ ]:
demographics = cj_data['demographics']
demographics.head()

In [ ]:
# Your code here

### The Join Type Changes Your Data

Here's the comparison, along with how many rows came back without any demographic information:

In [ ]:
inner = transactions.merge(demographics, on='household_id', how='inner')
left = transactions.merge(demographics, on='household_id', how='left')

print(f"transactions: {len(transactions):,} rows")
print(f"inner join:   {len(inner):,} rows")
print(f"left join:    {len(left):,} rows  ({left['age'].isnull().sum():,} with no demographics)")

Only 801 of 2,469 households filled out a demographic survey.

- **Inner** quietly drops more than **40%** of all transactions.
- **Left** keeps every transaction, but fills the missing demographics with `NaN`.

Neither one is wrong — but **you** have to choose, and know what you've chosen, before you summarize. **Check your row counts before and after every join.**

### 🎯 Your Turn: Join It

**Business Question:** Our marketing team wants to know which **age group** spends the most.

1. Join `transactions` and `demographics`. Which join type makes sense here?
2. For each `age` group, compute **total sales** and the **number of unique households**.
3. **Bonus:** add a `spend_per_household` column. Does the same age group still come out on top?

In [ ]:
# Your code here

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `df = df.rename(columns={'old': 'new'})` | Rename specific columns |
| `df.columns = df.columns.str.lower().str.replace(' ', '_')` | Clean **every** column name at once |
| `df = df.drop(columns=['a', 'b'])` | Remove columns you don't need |
| `df['new'] = df['a'] / df['b']` | Create a new column from existing ones |
| `df['col'].map({old: new})` | Translate values using a dict |
| `df.isnull().sum()` | Count missing values in each column |
| `df['col'] = df['col'].fillna(value)` | Fill missing values — once you know *why* they're missing |
| `df.agg({'col': ['sum', 'mean']})` | Several statistics across the whole table |
| `df.groupby('grp', as_index=False).agg({...})` | One row of statistics **per group** |
| `'nunique'` | Count **distinct** values (baskets, households) |
| `df.sort_values('col', ascending=False)` · `df.nlargest(n, 'col')` | Rank your results |
| `left.merge(right, on='key', how='inner')` | Join tables — `how` decides what happens to unmatched rows |

**The ideas underneath all of it:**

1. **Assign it back.** pandas returns new results — save them, or they're gone.
2. **Question → manipulate → summarize → join → interpret.** Most business questions use all three skills.
3. **Check what your join did.** Compare row counts before and after, every time.

## What's Next

- **Thursday Lab:** you'll play a junior analyst at RetailMax, answering questions from leadership with the Complete Journey data — manipulating, summarizing, and joining to find the answers.
- **Chapters to review:** Chapter 10 (Manipulating Data), Chapter 11 (Aggregating Data), Chapter 12 (Joining Data) — read these before Thursday's lab.